In [1]:
import time
import struct
import numpy as np
import scipy.ndimage
from pynq import Overlay, allocate

ol = Overlay("design_k5_wrapper.bit")
print("Overlay loaded")
print("IP blocks:", list(ol.ip_dict.keys()))

dma         = ol.axi_dma_0
conv_stream  = ol.conv2d_0
# fir_mmio_ip = ol.fir_mmio_0
hw_timer    = ol.axi_timer_0

Overlay loaded
IP blocks: ['conv2d_0', 'axi_timer_0', 'axi_dma_0', 'processing_system7_0']


In [2]:
inp_size = 64
kernel_size = 5
N_SAMPLES = inp_size * inp_size

img = np.random.randint(low=0, high=255, size=(inp_size, inp_size))/255
kernel = np.random.uniform(-1, 1, (kernel_size, kernel_size))
# kernel = [[0, 1, 0], [-1, 1, 1], [2, -2, 3]]
t0 = time.perf_counter()
exp_out = scipy.ndimage.convolve(img, kernel, mode='constant', cval=0.0)
t_now = time.perf_counter() - t0
t_now

0.004040716999952565

In [3]:
def float_to_raw_int(f_val):
    return struct.unpack('<I', struct.pack('<f', float(f_val)))[0]

In [4]:
in_buf  = allocate(shape=(N_SAMPLES,), dtype=np.float32)
out_buf = allocate(shape=(N_SAMPLES,), dtype=np.float32)

np.copyto(in_buf, list(img.flatten()))
out_buf[:] = 0

# 1. Write the 5x5 kernel weights using the helper function!
# Row 0
conv_stream.write(0x10, float_to_raw_int(kernel[0][0]))
conv_stream.write(0x18, float_to_raw_int(kernel[0][1]))
conv_stream.write(0x20, float_to_raw_int(kernel[0][2]))
conv_stream.write(0x28, float_to_raw_int(kernel[0][3]))
conv_stream.write(0x30, float_to_raw_int(kernel[0][4]))

# Row 1
conv_stream.write(0x38, float_to_raw_int(kernel[1][0]))
conv_stream.write(0x40, float_to_raw_int(kernel[1][1]))
conv_stream.write(0x48, float_to_raw_int(kernel[1][2]))
conv_stream.write(0x50, float_to_raw_int(kernel[1][3]))
conv_stream.write(0x58, float_to_raw_int(kernel[1][4]))

# Row 2
conv_stream.write(0x60, float_to_raw_int(kernel[2][0]))
conv_stream.write(0x68, float_to_raw_int(kernel[2][1]))
conv_stream.write(0x70, float_to_raw_int(kernel[2][2]))
conv_stream.write(0x78, float_to_raw_int(kernel[2][3]))
conv_stream.write(0x80, float_to_raw_int(kernel[2][4]))

# Row 3
conv_stream.write(0x88, float_to_raw_int(kernel[3][0]))
conv_stream.write(0x90, float_to_raw_int(kernel[3][1]))
conv_stream.write(0x98, float_to_raw_int(kernel[3][2]))
conv_stream.write(0xa0, float_to_raw_int(kernel[3][3]))
conv_stream.write(0xa8, float_to_raw_int(kernel[3][4]))

# Row 4
conv_stream.write(0xb0, float_to_raw_int(kernel[4][0]))
conv_stream.write(0xb8, float_to_raw_int(kernel[4][1]))
conv_stream.write(0xc0, float_to_raw_int(kernel[4][2]))
conv_stream.write(0xc8, float_to_raw_int(kernel[4][3]))
conv_stream.write(0xd0, float_to_raw_int(kernel[4][4]))

# 2. Write the 1D image size to the NEW offset (0xd8)
conv_stream.write(0xd8, inp_size)

# 3. Start the IP
conv_stream.write(0x00, 0x01)   # ap_start

# Trigger DMA transfer
t0 = time.perf_counter()
dma.sendchannel.transfer(in_buf)
dma.recvchannel.transfer(out_buf)
dma.sendchannel.wait()
dma.recvchannel.wait()
t_dma = time.perf_counter() - t0

y_dma = np.array(out_buf, dtype=np.float32)
print(f"DMA: {N_SAMPLES} samples in {t_dma*1e3:.2f} ms "
      f"({t_dma/N_SAMPLES*1e6:.1f} us/sample)")

DMA: 4096 samples in 2.93 ms (0.7 us/sample)


In [5]:
print("y_dma:", y_dma)

y_dma: [ 0.1288715   0.61104906  1.0852128  ... -3.5288992  -3.4068334
 -3.078487  ]


In [6]:
np.max(np.abs(y_dma - exp_out.flatten())) * 255

0.0002684699141197733

In [7]:
y_dma.shape

(4096,)

In [8]:
TCSR0, TLR0, TCR0 = 0x00, 0x04, 0x08
FCLK_MHZ = 100.0

def timer_start(tmr):
    tmr.write(TLR0, 0)
    tmr.write(TCSR0, 0x020)   # load
    tmr.write(TCSR0, 0x080)   # enable, count up

def timer_stop(tmr):
    cycles = tmr.read(TCR0)
    tmr.write(TCSR0, 0x000)
    return cycles

In [9]:

# 3. Start the IP
conv_stream.write(0x00, 0x01)   # ap_start
dma.sendchannel.start()
dma.recvchannel.start()
# Trigger DMA transfer
timer_start(hw_timer)
dma.sendchannel.transfer(in_buf)
dma.recvchannel.transfer(out_buf)
dma.sendchannel.wait()
dma.recvchannel.wait()
cycles = timer_stop(hw_timer)

y_dma = np.array(out_buf, dtype=np.float32)

total_us = cycles / FCLK_MHZ
per_sample_us = total_us / N_SAMPLES

print(f"HW timer: {cycles} cycles = {total_us:.3f} us total @ {FCLK_MHZ:.0f} MHz")
print(f"HW: {N_SAMPLES} samples in {total_us/1e3:.2f} ms ({per_sample_us:.3f} us/sample)")

HW timer: 210621 cycles = 2106.210 us total @ 100 MHz
HW: 4096 samples in 2.11 ms (0.514 us/sample)


In [10]:
np.max(np.abs(y_dma - exp_out.flatten())) * 255

0.0002684699141197733